In [1]:
from transformers.models import smolvlm

/storage/project/r-cj124-0/sibidapo3/anxcnda/aeroduo/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from high_uav.dataset import AeroduoDataset, collate_fn
from torch.utils.data import DataLoader
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from typing import Dict, List, Optional, Tuple

/storage/project/r-cj124-0/sibidapo3/8750/aeroduo_ws/aeroduo/pilot_llm/Grounded-SAM-2/sam2/modeling/sam/transformer.py:23: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()


In [3]:
dataset_root = "/storage/project/r-cj124-0/sibidapo3/8750/aeroduo_ws/aeroduo/data/Hal-13k"
window_T = 5
action_horizon = 8

_POLICY_KEYS = frozenset({
    "bev_images",
    "high_uav_poses",
    "low_uav_poses_window",
    "low_uav_pose_current",
    "low_uav_traj_target",
    "instruction",
})

In [4]:
train_dataset = AeroduoDataset(
    dataset_root=dataset_root,
    window_T=window_T,
    action_horizon=action_horizon,
)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

In [5]:
for _step, batch in enumerate(train_dataloader):
    policy_inputs: Dict = {k: v for k, v in batch.items() if k in _POLICY_KEYS}
    if _step > 1:
        break

In [6]:
policy_inputs.keys()

dict_keys(['bev_images', 'high_uav_poses', 'low_uav_poses_window', 'low_uav_pose_current', 'low_uav_traj_target', 'instruction'])

In [7]:
from high_uav.bev_segmentation import load_models
from high_uav.bev_encoder import BEVEncoder

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
sam2_predictor, grounding_model, _ = load_models(device=device)
bev_encoder = BEVEncoder(sam2_predictor, grounding_model)


bev_images = policy_inputs['bev_images']
high_uav_poses = policy_inputs['high_uav_poses']
low_uav_pose_window = policy_inputs['low_uav_poses_window']
low_uav_pose_current = policy_inputs['low_uav_pose_current']
low_uav_traj_target = policy_inputs['low_uav_traj_target']
instruction = policy_inputs['instruction']

B = len(instruction)
T = len(bev_images[0])
dtype = np.float32

all_image_embeds: List[torch.Tensor]     = []   # B × Tensor[T,256,64,64]
all_masks:        List[List]             = []   # B × T × ndarray[N,H,W]
all_detections:   List[List]             = []   # B × T × List[dict]
all_hidden:       List[torch.Tensor]     = []   # B × Tensor[T, S, 2048]

for b in range(B):
    # BEVEncoder: one Hiera forward for all T frames in this episode
    img_embs_b, masks_b, dets_b = bev_encoder(
        bev_images[b], instruction[b], device
    )   # img_embs_b: [T, 256, 64, 64]
    all_image_embeds.append(img_embs_b)
    all_masks.append(masks_b)
    all_detections.append(dets_b)

In [9]:
len(all_image_embeds[0])

5

In [10]:
all_image_embeds[0].shape

torch.Size([5, 256, 64, 64])

In [11]:
all_masks[0][4].shape

(5, 1024, 1024)

In [12]:
len(all_detections[0][4])

5

In [13]:
class UAVPoseProjector(nn.Module):
    """
    Project a UAV state into SmolVLM2 token space.

    Uses the first four elements (x, y, z, heading_rad); heading is encoded
    as (sin, cos), giving a fixed 5-dim input to a linear layer regardless of
    the full state vector length.

    Input  : Tensor [B, >=4]
    Output : Tensor [B, 1, vlm_hidden_dim]
    """

    def __init__(self, uav_pose_dim: int = 5, vlm_hidden_dim: int = 2048):
        super().__init__()
        self.proj = nn.Linear(uav_pose_dim, vlm_hidden_dim)

    def _encode_heading(self, pose: torch.Tensor) -> torch.Tensor:
        if pose.dim() == 1:
            pose = pose.unsqueeze(0)
        xyz   = pose[:, :3]
        sin_h = torch.sin(pose[:, 3]).unsqueeze(-1)
        cos_h = torch.cos(pose[:, 3]).unsqueeze(-1)
        return torch.cat([xyz, sin_h, cos_h], dim=-1)  # [B, 5]

    def forward(self, pose: torch.Tensor) -> torch.Tensor:
        encoded = self._encode_heading(pose)       # [B, 5]
        token   = self.proj(encoded)               # [B, vlm_hidden_dim]
        return token.unsqueeze(1)                  # [B, 1, vlm_hidden_dim]

In [14]:
from high_uav.config import AeroduoConfig
cfg = AeroduoConfig(
    window_T=5, action_horizon=8,
    mixed_precision="bf16"
)

In [15]:
from transformers import AutoModelForImageTextToText, AutoProcessor

cfg = cfg

processor = AutoProcessor.from_pretrained(cfg.smolvlm2_model_name)
processor.image_processor.do_image_splitting = False

_dtype_map = {
    "bfloat16": torch.bfloat16,
    "float16":  torch.float16,
    "float32":  torch.float32,
}
load_dtype = _dtype_map.get(
    getattr(cfg, "smolvlm2_load_dtype", "bfloat16"), torch.bfloat16
)

vlm = AutoModelForImageTextToText.from_pretrained(
    cfg.smolvlm2_model_name,
    torch_dtype=load_dtype,
).to("cuda")
vlm.eval()

for param in vlm.parameters():
    param.requires_grad_(False)


You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.
Loading checkpoint shards: 100%|██████████| 2/2 [00:40<00:00, 20.39s/it]


In [106]:
_lm_layers = vlm.model.text_model.layers
n = len(_lm_layers)
cutoff = cfg.vlm_layer_cutoff if cfg.vlm_layer_cutoff is not None else n // 2
vlm_layer_cutoff = cutoff
vlm.model.text_model.layers = _lm_layers[:cutoff]

hidden_size = vlm.config.text_config.hidden_size
pose_token_proj = UAVPoseProjector(uav_pose_dim=5, vlm_hidden_dim=hidden_size).to("cuda")
# for param in pose_token_proj.parameters():
#     param.requires_grad_(False)

In [107]:
trainable = sum(p.numel() for p in pose_token_proj.parameters() if p.requires_grad)
total = sum(p.numel() for p in pose_token_proj.parameters())
print(f"trainable params: {trainable:,} || all params: {total:,} || trainable%: {100 * trainable / total:.4f}")


trainable params: 12,288 || all params: 12,288 || trainable%: 100.0000


In [108]:
def embed_image(
    vlm,
    pixel_values: torch.Tensor,
    pixel_attention_mask: torch.Tensor,
) -> torch.Tensor:
    pixel_values = pixel_values.to(dtype=vlm.model.vision_model.dtype)
    return vlm.get_image_features(
        pixel_values=pixel_values,
        pixel_attention_mask=pixel_attention_mask,
    )

def embed_tokens(vlm, token_ids: torch.Tensor) -> torch.Tensor:
    return vlm.model.text_model.get_input_embeddings()(token_ids)


def build_processor_inputs(
        processor,
        bev_image,
        language_text: str,
        device: torch.device = torch.device("cuda"),
    ) -> dict:
        text_parts = language_text.split(
            "The description of the target and its surrounding is shown below."
        )
        direction_text = text_parts[0].strip().split(
            "Compass north corresponds to the top of the bird's-eye-view image."
        )[-1].strip()
        target_text = text_parts[-1].strip()

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": bev_image},
                    {
                        "type": "text",
                        "text": (
                            "North-aligned bird's-eye-view aerial image.\n"
                            f"Directionl prior: {direction_text}\n"
                            f"Target: {target_text}\n"
                            "Identify scene regions, landmarks and sturctures "
                            "relevant for locating the target and guiding navigation toward it."
                        ),
                    },
                ],
            }
        ]
        prompt = processor.apply_chat_template(messages, add_generation_prompt=False)
        inputs = processor(text=prompt, images=[bev_image], return_tensors="pt")
        return {k: v.to(device) for k, v in inputs.items()}



In [109]:
bev = bev_images[0][0]
desc = instruction[0]
low_state = low_uav_pose_window[0][0].to(device)
high_state = high_uav_poses[0][0].to(device)

In [110]:
low_state_token  = pose_token_proj(low_state).to(device).to(torch.bfloat16)
high_state_token = pose_token_proj(high_state).to(device).to(torch.bfloat16)


proc = build_processor_inputs(
    processor, bev, desc, device
)
input_ids            = proc["input_ids"]
pixel_values         = proc["pixel_values"]
pixel_attention_mask = proc["pixel_attention_mask"]
attn_mask            = proc["attention_mask"]

inputs_embeds = embed_tokens(vlm, input_ids)

In [111]:
image_embeds  = embed_image(vlm, pixel_values, pixel_attention_mask)

In [112]:
inputs_embeds.shape

torch.Size([1, 192, 2048])

In [113]:
inputs_embeds = vlm.model.inputs_merger(
            input_ids=input_ids,
            inputs_embeds=inputs_embeds,
            image_hidden_states=image_embeds,
        )

In [114]:
image_embeds.shape

torch.Size([1, 81, 2048])

In [115]:
inputs_embeds.shape

torch.Size([1, 192, 2048])

In [116]:
attn_mask.shape

torch.Size([1, 192])

In [117]:
pixel_attention_mask.shape

torch.Size([1, 1, 384, 384])

In [118]:
inputs_embeds = torch.cat(
    [inputs_embeds, high_state_token, low_state_token],
    dim=1,
)
inputs_embeds.shape

torch.Size([1, 194, 2048])

In [119]:
attn_mask.shape

torch.Size([1, 192])

In [120]:
attention_mask = torch.cat(
            [attn_mask, torch.ones((1, 2), dtype=attn_mask.dtype, device=device)],
            dim=1,
        )

In [121]:
lm_out = vlm.model.text_model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            output_hidden_states=False,
            return_dict=True,
            use_cache=False,
        )

In [122]:
lm_out

BaseModelOutputWithPast(last_hidden_state=tensor([[[-7.5378e-03, -8.6670e-03,  3.3722e-03,  ...,  2.0215e-01,
          -7.8125e-03,  1.0132e-02],
         [-9.2188e-01, -4.8828e-02, -2.3828e-01,  ..., -1.2734e+00,
          -9.8047e-01, -1.6504e-01],
         [ 3.3398e-01,  5.5469e-01,  2.0215e-01,  ..., -1.9922e+00,
          -7.8906e-01, -6.2500e-01],
         ...,
         [-1.1816e-01, -6.9531e-01,  4.5312e-01,  ..., -9.8125e+00,
          -9.3262e-02, -5.9766e-01],
         [ 3.6719e-01,  3.7109e-01,  3.6719e-01,  ...,  8.5938e-02,
           8.7891e-02, -2.6367e-01],
         [ 1.1016e+00,  6.0156e-01,  3.0273e-01,  ...,  1.7266e+00,
           1.2598e-01, -3.7500e-01]]], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<MulBackward0>), past_key_values=None, hidden_states=None, attentions=None)

In [123]:
lm_out.last_hidden_state.requires_grad

True